# MicroStrategy REST Admin

## Load libraries

In [11]:
import yaml
import json
from mstrio.object_management import folder
from mstrio.api import metrics,browsing,filters,attributes,transformations

from mstr_robotics.mstr_classes import MdSearches
from mstr_robotics._connectors import MstrApi

from time import sleep


all_comp_obj_d_l=[]
i_md_searches=MdSearches()
i_mstr_api=MstrApi()

In [12]:
from mstr_robotics.mstr_classes import get_conn
with open('..\\config\\user_d.yml', 'r') as openfile:
    user_d = yaml.safe_load(openfile)

conn_params =  user_d["conn_params"]
conn = get_conn(**conn_params)
conn.headers['Content-type'] = "application/json"
project_id=user_d["conn_params"]["project_id"]
conn.select_project(project_id)

Connection to Strategy One Intelligence Server has been established.
No project selected.


## Endpoints

In [13]:
def get_child_objects(conn,object_id):
    # Both helpers are delegated to the mstr_robotics package:
    # - i_mstr_api.get_proj_obj_by_id_l: resolves the object's type/subtype/name
    #   from just the id (same searches/objects endpoint the old get_obj_details used).
    # - i_md_searches.search_for_used_in_obj_direct: wraps store_search_instance +
    #   paged get_search_results to find the dependents, handling the 0-results case.
    obj_det_l=i_mstr_api.get_proj_obj_by_id_l(conn,[object_id],org_def_fg=True)
    if not obj_det_l:
        return []
    object_det_d=obj_det_l[0]
    obj_l=[{"id":object_det_d["id"],"type":object_det_d["type"]}]
    dpn_rows=i_md_searches.search_for_used_in_obj_direct(
        conn,obj_l,dpn_fg=True,info_level="base",count_only_fg=False)
    dpn_child_d_l=[]
    for row in dpn_rows:
        # objects with no dependents yield a dummy row without dpn_ keys; skip it
        if not row.get("dpn_id"):
            continue
        dpn_child_d={}
        dpn_child_d["id"]=object_det_d["id"]
        dpn_child_d["type"]=object_det_d["type"]
        dpn_child_d["subtype"]=object_det_d["subtype"]
        dpn_child_d["name"]=object_det_d["name"]
        dpn_child_d["child_dpn_id"]=row["dpn_id"]
        dpn_child_d["child_dpn_type"]=row["dpn_type"]
        dpn_child_d["child_dpn_subtype"]=row["dpn_subtype"]
        dpn_child_d["child_dpn_name"]=row["dpn_name"]
        dpn_child_d_l.append(dpn_child_d.copy())
    return dpn_child_d_l

In [ ]:
#object ids are maintained in ..\config\jupyter_objects_d.yml
with open("..\\config\\jupyter_objects_d.yml", "r") as openfile:
    jupyter_objects_d = yaml.safe_load(openfile)
nb_d = jupyter_objects_d["semantic_endpoints"]

#metric
metric_d_l=[]
metric_id=nb_d["misc"]["metric_id"]
metric_l=[metric_id]
for metric_id in metric_l:
    metric_def=metrics.get_metric(
        connection=conn,
        id=metric_id,
        changeset_id=None,
        show_expression_as="tokens",
        show_filter_tokens=False
    )
    metric_def=metric_def.json()
    metric_d={}
    metric_d["project_id"]=conn.project_id
    metric_d["id"]=metric_id
    metric_d["name"]=metric_def["name"]
    metric_d["text"]=metric_def["expression"]["text"]
    metric_d["expression"]=metric_def["expression"]
    metric_d["transformations_l"]=None
    metric_d["commplexity"]=1
    metric_d_l.append(metric_d.copy())
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=metric_id))

metric_d_l


In [ ]:
#filter
filter_d_l=[]
filter_l=nb_d["misc"]["filter_l"]
for filter_id in filter_l:
    filter_def=filters.get_filter(connection=conn, id=filter_id, project_id=project_id).json()
    filter_d={}
    filter_d["project_id"]=conn.project_id
    filter_d["id"]=filter_id
    filter_d["name"]=filter_def["name"]
    filter_d["text"]=filter_def["qualification"]["text"]
    filter_d["filter_type"]=filter_def["qualification"]["tree"]["type"]
    filter_d["commplexity"]=1
    filter_d["qualification"]=filter_def["qualification"]
    filter_d_l.append(filter_d.copy())
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=filter_id))
filter_d_l

In [ ]:
#attributes
attribute_d_l=[]   
att_parent_child_d_l=[] 
attribute_l=nb_d["misc"]["attribute_l"]
attribute_def=attributes.get_attribute(connection=conn,id=attribute_l[0],show_expression_as="tokens").json()

for att in attribute_l:
    attribute_d={}
    attribute_d["project_id"]=conn.project_id
    attribute_def=attributes.get_attribute(connection=conn,id=att,show_expression_as="tokens").json()
    attribute_d["id"]=attribute_def["id"]
    attribute_d["name"]=attribute_def["name"]
    #attribute_d["text"]=attribute_def["expression"]["text"]
    attribute_d["def"]=attribute_def
    attribute_d_l.append(attribute_d.copy())


for r in attribute_def["relationships"]:
  
    att_parent_child_d={}

    if attribute_def["id"]!=r["parent"]["objectId"]:

        att_parent_child_d["rel_attribute_id"]=r["parent"]["objectId"]
        att_parent_child_d["rel_attribute_name"]=r["parent"]["name"]
        att_parent_child_d["rel_table"]=r["relationshipTable"]["name"]
        att_parent_child_d["rel_type"]=r["relationshipType"]
        att_parent_child_d["type"]="parent"
    else:

        att_parent_child_d["rel_attribute_id"]=r["child"]["objectId"]
        att_parent_child_d["rel_attribute_name"]=r["child"]["name"]
        att_parent_child_d["rel_table"]=r["relationshipTable"]["name"]
        att_parent_child_d["rel_type"]=r["relationshipType"]
        att_parent_child_d["type"]="child"
    att_parent_child_d_l.append(att_parent_child_d.copy())
att_parent_child_d_l

In [17]:
#facts

In [ ]:
#transformations
transformation_d_l=[]   
transformation_l=nb_d["misc"]["transformation_l"]
for t in transformation_l:
    transformation_d={}
    trans_def=transformations.get_transformation(connection=conn,id=t).json()
    transformation_d["id"]=trans_def["id"]
    transformation_d["name"]=trans_def["name"]
    transformation_d["mapping_type"]=trans_def["mappingType"]
    transformation_d["def"]=trans_def
    transformation_d_l.append(transformation_d.copy())


transformation_d_l

In [ ]:
#Olap reports
report_d_l=[]
report_l=nb_d["reports"]["report_l"]
for report_id in report_l:
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=report_id))

In [ ]:
#OlapCubes
olap_cube_l=nb_d["cubes"]["olap_cube_l"]
report_l=nb_d["reports"]["report_l"]
for cube_id in olap_cube_l:
    all_comp_obj_d_l.append(get_child_objects(conn,object_id=cube_id))